In [4]:
# Installer les bibliothèques nécessaires
!pip install simpy
!pip install geopy
!pip install pandas openpyxl scikit-learn ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 12.1 MB/s eta 0:00:00


In [5]:
# Importer les bibliothèques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import simpy
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import geopy.distance
from sklearn.linear_model import LinearRegression

In [18]:
# Coordonnées géographiques des distributeurs
data_coords = {
    'CASA': (33.5731, -7.5898),
    'beni-mellal': (32.3454, -6.3549),
    'Eljadida': (33.2548, -8.4629),
    'Fes': (34.0372, -5.0081),
    'Kenitra': (34.2610, -6.5803),
    'Marrakech': (31.6296, -7.9811),
    'Meknes': (33.8949, -5.5555),
    'Midelt': (32.6868, -4.7488),
    'Nador': (35.1740, -2.9285),
    'Oujda': (34.6822, -1.9076),
    'Rabat': (34.0209, -6.8442),
    'Settat': (33.0293, -7.6399),
    'Tanger': (35.7654, -5.8070),
    'Tetouane': (35.5741, -5.3716),
    'AGADIR': (30.4278, -9.5992)
}

In [17]:

# Charger
uploaded = files.upload()
filename = next(iter(uploaded.keys()))

# Forcer header=0
data = pd.read_excel(filename, sheet_name=None, header=0)

# Vérifier
first_sheet = list(data.keys())[0]
print(f"Colonnes dans {first_sheet}: {data[first_sheet].columns.tolist()}")
print(f"\nAperçu:\n{data[first_sheet].head(3)}")

# Menus simples
dist_dd = widgets.Dropdown(options=list(data.keys()), description='Distributeur:')
brand_dd = widgets.Dropdown(options=[], description='Marque:')
fam_dd = widgets.Dropdown(options=[], description='Famille:')

def maj_brands(c):
    df = data[dist_dd.value]
    if 'Brand' in df.columns:
        brand_dd.options = df['Brand'].dropna().unique().tolist()

def maj_familles(c):
    df = data[dist_dd.value]
    if brand_dd.value and 'Famille' in df.columns:
        fam_dd.options = df[df['Brand'] == brand_dd.value]['Famille'].dropna().unique().tolist()

dist_dd.observe(maj_brands, names='value')
brand_dd.observe(maj_familles, names='value')

display(dist_dd, brand_dd, fam_dd)

# Initialiser
if list(data.keys()):
    dist_dd.value = list(data.keys())[0]

Saving donnees_distributeurs.xlsx to donnees_distributeurs (3).xlsx
Colonnes dans CASA: ['Brand', 'Famille', 'Juin', 'Juil', 'Août', 'Sept', 'Oct', 'Nov', 'Déc', 'Janv', 'Fév', 'Mars', 'Avril', 'Mai', 'TOTAL', 'demande moyenne', 'Ecart-type', 'Stock max (TN)', 'Stock min (TN)', 'Stock d alerte (TN)', 'Stock de sécurité (TN)', 'Périodicité de commande (jours)']

Aperçu:
       Brand Famille  Juin  Juil  Août  Sept  Oct  Nov  Déc  Janv  ...  Avril  \
0  Coca-Cola    Soda   272   362   503   375  176  214  196   168  ...    255   
1  Coca-Cola   Light   300   362   543   309  285  217  200   151  ...    265   
2      Pepsi    Soda   710   802   661   585  646  369  372   326  ...    477   

   Mai  TOTAL  demande moyenne  Ecart-type  Stock max (TN)  Stock min (TN)  \
0  169   3026       252.166667  155.988982             378              56   
1  232   3404       283.666667  151.167126             425              63   
2  588   6370       530.833333  226.444392             796           

Dropdown(description='Distributeur:', options=('CASA', 'Rabat', 'Tanger', 'Marrakech', 'Fes', 'Kenitra', 'Agad…

Dropdown(description='Marque:', options=(), value=None)

Dropdown(description='Famille:', options=(), value=None)

In [24]:
# Version simplifiée et robuste
analyse_btn = widgets.Button(description="📊 ANALYSER LA DISTRIBUTION",
                             button_style='success',
                             layout=widgets.Layout(width='300px', height='40px'))
output = widgets.Output()

def analyser(b):
    with output:
        output.clear_output()
        display(dist_dd, brand_dd, fam_dd, analyse_btn)

        dist = dist_dd.value
        marque = brand_dd.value
        famille = fam_dd.value

        if not all([dist, marque, famille]):
            print("❌ Veuillez sélectionner un distributeur, une marque et une famille")
            return

        # Récupérer les données
        df = data[dist]

        # Liste des mois
        mois = ['Juin', 'Juil', 'Août', 'Sept', 'Oct', 'Nov', 'Déc',
                'Janv', 'Fév', 'Mars', 'Avril', 'Mai']

        # Filtrer les colonnes qui existent
        mois_existants = [m for m in mois if m in df.columns]

        if not mois_existants:
            print("❌ Colonnes de ventes non trouvées")
            print(f"Colonnes disponibles: {df.columns.tolist()}")
            return

        # Filtrer les données
        mask = (df['Brand'] == marque) & (df['Famille'] == famille)
        ventes = df.loc[mask, mois_existants]

        if ventes.empty:
            print("❌ Aucune donnée pour cette combinaison")
            return

        # Aplatir les données
        valeurs = ventes.values.flatten()

        # Créer une figure avec 2 graphiques
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))

        # Histogramme
        axes[0].hist(valeurs, bins=8, color='skyblue', edgecolor='black', alpha=0.7)
        axes[0].axvline(valeurs.mean(), color='red', linewidth=2, label=f'Moyenne: {valeurs.mean():.1f}')
        axes[0].axvline(valeurs.mean() + valeurs.std(), color='orange', linestyle='--', label=f'+1σ: {valeurs.mean() + valeurs.std():.1f}')
        axes[0].axvline(valeurs.mean() - valeurs.std(), color='orange', linestyle='--', label=f'-1σ: {valeurs.mean() - valeurs.std():.1f}')
        axes[0].set_title(f'Distribution des ventes\n{marque} - {famille}', fontsize=12)
        axes[0].set_xlabel('Ventes (TN)')
        axes[0].set_ylabel('Fréquence')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Q-Q Plot
        stats.probplot(valeurs, dist="norm", plot=axes[1])
        axes[1].set_title('Q-Q Plot (Test de normalité)', fontsize=12)
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        # Statistiques descriptives
        print("\n📈 STATISTIQUES DESCRIPTIVES:")
        print(f"   • Nombre d'observations: {len(valeurs)}")
        print(f"   • Minimum: {valeurs.min():.2f}")
        print(f"   • Maximum: {valeurs.max():.2f}")
        print(f"   • Moyenne: {valeurs.mean():.2f}")
        print(f"   • Médiane: {np.median(valeurs):.2f}")
        print(f"   • Écart-type: {valeurs.std():.2f}")

        # Test de normalité (si assez de données)
        if 3 <= len(valeurs) <= 5000:
            stat, p = stats.shapiro(valeurs)
            print(f"\n🔬 TEST DE SHAPIRO-WILK:")
            print(f"   • Statistique W: {stat:.6f}")
            print(f"   • p-value: {p:.6f}")
            if p > 0.05:
                print("   ✅ Conclusion: Distribution normale (p > 0.05)")
            else:
                print("   ❌ Conclusion: Distribution non normale (p < 0.05)")

analyse_btn.on_click(analyser)
display(analyse_btn, output)

Button(button_style='success', description='📊 ANALYSER LA DISTRIBUTION', layout=Layout(height='40px', width='3…

Output()

In [ ]:
# Bouton pour la simulation simple
simulation_button = widgets.Button(description='Lancer la simulation simple')
output_simulation = widgets.Output()

# Fonction pour simuler une commande
def simulate_order(mean_demand, std_dev):
    quantity = abs(np.random.normal(mean_demand, std_dev))
    return max(1, int(np.round(quantity)))

def supply_chain_simple(env, distributor_data, selected_distributor, selected_brand, selected_family):
    # Paramètres de stock
    initial_stock = distributor_data['Stock max (TN)'].values[0]
    stock_alert = distributor_data['Stock d alerte (TN)'].values[0]
    stock_current = initial_stock

    # Historique
    stock_history = []
    order_logs = []
    satisfied_orders = 0
    unsatisfied_orders = 0
    total_sales = 0

    # Paramètres de demande
    mean_demand_month = distributor_data['demande moyenne'].values[0]
    std_dev_month = distributor_data['Ecart-type'].values[0]
    mean_demand_daily = mean_demand_month / 30
    std_dev_daily = std_dev_month / np.sqrt(30)

    for day in range(30):
        stock_history.append(stock_current)

        # Vérifier si réapprovisionnement nécessaire
        if stock_current <= stock_alert:
            replenishment_quantity = initial_stock - stock_current
            stock_current += replenishment_quantity
            order_logs.append(f"Jour {day}: RÉAPPROVISIONNEMENT - {replenishment_quantity} TN livrés. Nouveau stock = {stock_current:.2f} TN")

        # Simuler une commande
        quantity_requested = simulate_order(mean_demand_daily, std_dev_daily)

        if quantity_requested <= stock_current:
            satisfied_orders += 1
            total_sales += quantity_requested
            stock_current -= quantity_requested
            order_logs.append(f"Jour {day}: Commande satisfaite - {quantity_requested} TN. Stock = {stock_current:.2f} TN")
        else:
            unsatisfied_orders += 1
            order_logs.append(f"Jour {day}: Commande NON satisfaite - {quantity_requested} TN demandées, Stock = {stock_current:.2f} TN")

        yield env.timeout(1)

    return satisfied_orders, unsatisfied_orders, total_sales, stock_history, order_logs, mean_demand_daily, std_dev_daily

def run_simulation_simple(b):
    with output_simulation:
        clear_output()
        display(distributeur_dropdown, brand_dropdown, famille_dropdown, simulation_button)

        selected_distributeur = distributeur_dropdown.value
        selected_brand = brand_dropdown.value
        selected_famille = famille_dropdown.value

        # Vérifications
        if not selected_brand or not selected_famille:
            print("Veuillez sélectionner une marque et une famille")
            return

        # Filtrer les données
        distributor_data = data[selected_distributeur]
        distributor_data = distributor_data[distributor_data['Brand'] == selected_brand]
        distributor_data = distributor_data[distributor_data['Famille'] == selected_famille]

        if distributor_data.empty:
            print("Aucune donnée disponible")
            return

        # Lancer la simulation
        env = simpy.Environment()
        process = env.process(supply_chain_simple(env, distributor_data, selected_distributeur, selected_brand, selected_famille))
        env.run()

        satisfied, unsatisfied, total_sales, stock_history, logs, mean_daily, std_daily = process.value

        # Visualisation
        plt.figure(figsize=(12, 6))
        plt.plot(stock_history, label=selected_distributeur, linewidth=2)
        plt.axhline(y=distributor_data['Stock d alerte (TN)'].values[0], color='orange', linestyle='--', label='Stock d alerte')
        plt.axhline(y=distributor_data['Stock min (TN)'].values[0], color='red', linestyle='--', label='Stock min')
        plt.title('Évolution du Stock - Simulation Simple', fontsize=14)
        plt.xlabel('Jours')
        plt.ylabel('Stock (TN)')
        plt.legend()
        plt.grid(True)
        plt.show()

        # Résultats
        rupture_rate = (unsatisfied / 30) * 100
        print(f"\n=== RÉSULTATS DE LA SIMULATION ===")
        print(f"Distributeur: {selected_distributeur}")
        print(f"Marque: {selected_brand}, Famille: {selected_famille}")
        print(f"Stock initial: {distributor_data['Stock max (TN)'].values[0]} TN")
        print(f"Demande moyenne journalière: {mean_daily:.2f} TN")
        print(f"Écart-type journalier: {std_daily:.2f} TN")
        print(f"Commandes satisfaites: {satisfied}/30")
        print(f"Commandes non satisfaites: {unsatisfied}/30")
        print(f"Taux de rupture: {rupture_rate:.2f}%")
        print(f"Total ventes: {total_sales} TN")

        print("\n=== JOURNAL DES ÉVÉNEMENTS ===")
        for log in logs:
            print(log)

simulation_button.on_click(run_simulation_simple)
display(simulation_button, output_simulation)

In [25]:
# Bouton pour la simulation simple
simulation_button = widgets.Button(description='📊 LANCER LA SIMULATION SIMPLE',
                                   button_style='primary',
                                   layout=widgets.Layout(width='300px', height='40px'))
output_simulation = widgets.Output()

# Fonction pour simuler une commande
def simulate_order(mean_demand, std_dev):
    quantity = abs(np.random.normal(mean_demand, std_dev))
    return max(1, int(np.round(quantity)))

def supply_chain_simple(env, distributor_data, selected_distributor, selected_brand, selected_family):
    # Paramètres de stock
    initial_stock = distributor_data['Stock max (TN)'].values[0]
    stock_alert = distributor_data['Stock d alerte (TN)'].values[0]
    stock_min = distributor_data['Stock min (TN)'].values[0]
    stock_current = initial_stock

    # Historique
    stock_history = []
    order_logs = []
    satisfied_orders = 0
    unsatisfied_orders = 0
    total_sales = 0

    # Paramètres de demande
    mean_demand_month = distributor_data['demande moyenne'].values[0]
    std_dev_month = distributor_data['Ecart-type'].values[0]
    mean_demand_daily = mean_demand_month / 30
    std_dev_daily = std_dev_month / np.sqrt(30)

    print(f"📊 Simulation démarrée pour {selected_brand} - {selected_family}")
    print(f"   Stock initial: {initial_stock} TN")
    print(f"   Stock d'alerte: {stock_alert} TN")
    print(f"   Demande moyenne journalière: {mean_demand_daily:.2f} TN")
    print(f"   Écart-type journalier: {std_dev_daily:.2f} TN")
    print("-" * 50)

    for day in range(30):
        stock_history.append(stock_current)

        # Vérifier si réapprovisionnement nécessaire
        if stock_current <= stock_alert:
            replenishment_quantity = initial_stock - stock_current
            stock_current += replenishment_quantity
            order_logs.append(f"Jour {day:2d}: 📦 RÉAPPROVISIONNEMENT - {replenishment_quantity:.1f} TN livrés. Nouveau stock = {stock_current:.1f} TN")

        # Simuler une commande
        quantity_requested = simulate_order(mean_demand_daily, std_dev_daily)

        if quantity_requested <= stock_current:
            satisfied_orders += 1
            total_sales += quantity_requested
            stock_current -= quantity_requested
            order_logs.append(f"Jour {day:2d}: ✅ Commande satisfaite - {quantity_requested:.1f} TN. Stock = {stock_current:.1f} TN")
        else:
            unsatisfied_orders += 1
            order_logs.append(f"Jour {day:2d}: ❌ Commande NON satisfaite - {quantity_requested:.1f} TN demandées, Stock = {stock_current:.1f} TN")

        yield env.timeout(1)

    return satisfied_orders, unsatisfied_orders, total_sales, stock_history, order_logs, mean_demand_daily, std_dev_daily

def run_simulation_simple(b):
    with output_simulation:
        clear_output()
        display(dist_dd, brand_dd, fam_dd, simulation_button)

        selected_distributeur = dist_dd.value
        selected_brand = brand_dd.value
        selected_famille = fam_dd.value

        # Vérifications
        if not selected_brand or not selected_famille:
            print("❌ Veuillez sélectionner une marque et une famille")
            return

        print(f"🔍 Recherche des données pour: {selected_distributeur} - {selected_brand} - {selected_famille}")

        # Filtrer les données
        distributor_data = data[selected_distributeur]
        distributor_data = distributor_data[distributor_data['Brand'] == selected_brand]
        distributor_data = distributor_data[distributor_data['Famille'] == selected_famille]

        if distributor_data.empty:
            print("❌ Aucune donnée disponible pour cette combinaison")
            print(f"Vérifiez que '{selected_brand}' et '{selected_famille}' existent dans le fichier")
            return

        # Afficher les données trouvées
        print(f"✅ Données trouvées!")
        print(f"   Stock max: {distributor_data['Stock max (TN)'].values[0]} TN")
        print(f"   Stock min: {distributor_data['Stock min (TN)'].values[0]} TN")
        print(f"   Stock d'alerte: {distributor_data['Stock d alerte (TN)'].values[0]} TN")
        print(f"   Demande moyenne mensuelle: {distributor_data['demande moyenne'].values[0]:.1f} TN")
        print(f"   Écart-type mensuel: {distributor_data['Ecart-type'].values[0]:.1f} TN")
        print("-" * 50)

        # Lancer la simulation
        env = simpy.Environment()
        process = env.process(supply_chain_simple(env, distributor_data, selected_distributeur, selected_brand, selected_famille))
        env.run()

        satisfied, unsatisfied, total_sales, stock_history, logs, mean_daily, std_daily = process.value

        # Visualisation
        plt.figure(figsize=(14, 6))

        # Tracer l'évolution du stock
        plt.plot(stock_history, label=f'Stock - {selected_distributeur}', linewidth=2, color='blue')

        # Lignes de référence
        stock_alert_val = distributor_data['Stock d alerte (TN)'].values[0]
        stock_min_val = distributor_data['Stock min (TN)'].values[0]
        stock_max_val = distributor_data['Stock max (TN)'].values[0]

        plt.axhline(y=stock_alert_val, color='orange', linestyle='--', linewidth=2, label=f'Stock d alerte ({stock_alert_val} TN)')
        plt.axhline(y=stock_min_val, color='red', linestyle='--', linewidth=2, label=f'Stock min ({stock_min_val} TN)')
        plt.axhline(y=stock_max_val, color='green', linestyle=':', linewidth=2, label=f'Stock max ({stock_max_val} TN)')

        # Remplir les zones
        plt.fill_between(range(len(stock_history)), stock_min_val, stock_history,
                         where=(np.array(stock_history) > stock_min_val),
                         color='lightgreen', alpha=0.3, label='Zone sécurité')
        plt.fill_between(range(len(stock_history)), 0, stock_min_val,
                         color='lightcoral', alpha=0.3, label='Zone critique')

        plt.title(f'📈 Évolution du Stock - {selected_brand} {selected_famille} ({selected_distributeur})', fontsize=14)
        plt.xlabel('Jours', fontsize=12)
        plt.ylabel('Stock (TN)', fontsize=12)
        plt.legend(loc='upper right')
        plt.grid(True, alpha=0.3)
        plt.xticks(range(0, 31, 5))
        plt.tight_layout()
        plt.show()

        # Résultats
        rupture_rate = (unsatisfied / 30) * 100

        # Afficher les résultats dans un cadre stylisé
        from IPython.display import HTML

        result_html = f"""
        <div style="background-color: #f5f5f5; padding: 20px; border-radius: 10px; border: 2px solid #4CAF50;">
            <h3 style="color: #333; margin-top: 0;">📊 RÉSULTATS DE LA SIMULATION</h3>
            <table style="width: 100%; border-collapse: collapse;">
                <tr>
                    <td style="padding: 8px; background-color: #e0e0e0; width: 40%;"><b>Distributeur:</b></td>
                    <td style="padding: 8px;">{selected_distributeur}</td>
                </tr>
                <tr>
                    <td style="padding: 8px; background-color: #e0e0e0;"><b>Marque / Famille:</b></td>
                    <td style="padding: 8px;">{selected_brand} - {selected_famille}</td>
                </tr>
                <tr>
                    <td style="padding: 8px; background-color: #e0e0e0;"><b>Stock initial:</b></td>
                    <td style="padding: 8px;">{stock_max_val} TN</td>
                </tr>
                <tr>
                    <td style="padding: 8px; background-color: #e0e0e0;"><b>Demande moyenne journalière:</b></td>
                    <td style="padding: 8px;">{mean_daily:.2f} TN</td>
                </tr>
                <tr>
                    <td style="padding: 8px; background-color: #e0e0e0;"><b>Écart-type journalier:</b></td>
                    <td style="padding: 8px;">{std_daily:.2f} TN</td>
                </tr>
                <tr>
                    <td style="padding: 8px; background-color: #e0e0e0;"><b>Commandes satisfaites:</b></td>
                    <td style="padding: 8px;">{satisfied}/30</td>
                </tr>
                <tr>
                    <td style="padding: 8px; background-color: #e0e0e0;"><b>Commandes non satisfaites:</b></td>
                    <td style="padding: 8px;">{unsatisfied}/30</td>
                </tr>
                <tr>
                    <td style="padding: 8px; background-color: #e0e0e0;"><b>Taux de rupture:</b></td>
                    <td style="padding: 8px;"><span style="background-color: {'#FFB6C1' if rupture_rate > 10 else '#90EE90'}; padding: 3px 8px; border-radius: 5px;">{rupture_rate:.2f}%</span></td>
                </tr>
                <tr>
                    <td style="padding: 8px; background-color: #e0e0e0;"><b>Total ventes:</b></td>
                    <td style="padding: 8px;">{total_sales:.1f} TN</td>
                </tr>
            </table>
        </div>
        """

        display(HTML(result_html))

        # Afficher le journal des événements (10 premiers pour ne pas surcharger)
        print("\n📋 JOURNAL DES ÉVÉNEMENTS (10 premiers jours):")
        for i, log in enumerate(logs[:10]):
            print(log)

        if len(logs) > 10:
            print(f"... et {len(logs)-10} autres événements")

simulation_button.on_click(run_simulation_simple)
display(simulation_button, output_simulation)

Button(button_style='primary', description='📊 LANCER LA SIMULATION SIMPLE', layout=Layout(height='40px', width…

Output()

LA SIMULATION AVEC RÉSEAU

In [26]:
# Bouton pour la simulation avancée
reseau_button = widgets.Button(description='🌐 LANCER LA SIMULATION AVEC RÉSEAU',
                               button_style='warning',
                               layout=widgets.Layout(width='350px', height='40px'))
output_reseau = widgets.Output()

# Fonctions utilitaires pour le réseau
def calculate_distance(coord1, coord2):
    """Calcule la distance entre deux distributeurs"""
    return geopy.distance.geodesic(coord1, coord2).km

def find_nearest_distributor(selected_distributor, quantity_requested, data_coords):
    """Trouve le distributeur le plus proche avec stock suffisant"""
    nearest = None
    min_distance = float('inf')

    print(f"   🔍 Recherche d'un distributeur pour {quantity_requested:.1f} TN...")

    for dist, coords in data_coords.items():
        if dist != selected_distributor and dist in data:
            try:
                # Vérifier le stock de sécurité
                df_dist = data[dist]
                stock_security = df_dist['Stock de sécurité (TN)'].values[0]

                # Vérifier aussi le stock max disponible
                stock_max = df_dist['Stock max (TN)'].values[0]
                stock_disponible = min(stock_security, stock_max)

                if stock_disponible >= quantity_requested:
                    distance = calculate_distance(data_coords[selected_distributor], coords)
                    print(f"     → {dist}: {stock_disponible:.1f} TN dispo, distance {distance:.1f} km")
                    if distance < min_distance:
                        min_distance = distance
                        nearest = dist
            except Exception as e:
                continue

    if nearest:
        print(f"   ✅ Distributeur le plus proche trouvé: {nearest} ({min_distance:.1f} km)")
    else:
        print(f"   ❌ Aucun distributeur trouvé avec stock suffisant")

    return nearest

def supply_chain_reseau(env, distributor_data, data_coords, selected_distributor, selected_brand, selected_family):
    # Paramètres de stock
    initial_stock = distributor_data['Stock max (TN)'].values[0]
    stock_alert = distributor_data['Stock d alerte (TN)'].values[0]
    stock_min = distributor_data['Stock min (TN)'].values[0]
    stock_current = initial_stock

    # Historique
    stock_history = []
    nearest_history = []
    order_logs = []
    satisfied_orders = 0
    unsatisfied_orders = 0
    total_sales = 0
    transfers = 0

    # Paramètres de demande
    mean_demand_month = distributor_data['demande moyenne'].values[0]
    std_dev_month = distributor_data['Ecart-type'].values[0]
    mean_demand_daily = mean_demand_month / 30
    std_dev_daily = std_dev_month / np.sqrt(30)

    print(f"\n🌐 SIMULATION AVEC RÉSEAU DÉMARRÉE")
    print(f"   Distributeur principal: {selected_distributor}")
    print(f"   Produit: {selected_brand} - {selected_family}")
    print(f"   Stock initial: {initial_stock} TN")
    print(f"   Stock d'alerte: {stock_alert} TN")
    print(f"   Demande moyenne journalière: {mean_demand_daily:.2f} TN")
    print("=" * 60)

    for day in range(30):
        stock_history.append(stock_current)

        # Vérifier si réapprovisionnement nécessaire (tous les 7 jours par exemple)
        if day > 0 and day % 7 == 0:
            replenishment = initial_stock - stock_current
            if replenishment > 0:
                stock_current += replenishment
                order_logs.append(f"Jour {day:2d}: 📦 RÉAPPROVISIONNEMENT - {replenishment:.1f} TN livrés")

        # Simuler une commande
        quantity_requested = simulate_order(mean_demand_daily, std_dev_daily)

        if quantity_requested <= stock_current:
            satisfied_orders += 1
            total_sales += quantity_requested
            stock_current -= quantity_requested
            order_logs.append(f"Jour {day:2d}: ✅ Commande satisfaite - {quantity_requested:.1f} TN. Stock = {stock_current:.1f} TN")
            nearest_history.append(selected_distributor)
        else:
            unsatisfied_orders += 1
            order_logs.append(f"Jour {day:2d}: ❌ RUPTURE - {quantity_requested:.1f} TN demandées, Stock = {stock_current:.1f} TN")

            # Chercher un autre distributeur
            nearest = find_nearest_distributor(selected_distributor, quantity_requested, data_coords)

            if nearest:
                try:
                    # Récupérer le stock disponible chez l'autre distributeur
                    df_nearest = data[nearest]
                    stock_dispo = df_nearest['Stock max (TN)'].values[0]
                    stock_secu = df_nearest['Stock de sécurité (TN)'].values[0]

                    # Quantité transférable (limitée par le stock dispo et la demande)
                    transfer_qty = min(quantity_requested, stock_dispo, stock_secu)

                    if transfer_qty > 0:
                        stock_current += transfer_qty
                        stock_current -= quantity_requested  # Satisfaire la commande
                        total_sales += quantity_requested
                        satisfied_orders += 1  # Finalement satisfaite
                        unsatisfied_orders -= 1  # Corriger le compteur
                        transfers += 1

                        order_logs.append(f"     → 🔄 Transfert de {transfer_qty:.1f} TN depuis {nearest}")
                        order_logs.append(f"     → ✅ Commande satisfaite après transfert")
                        nearest_history.append(nearest)
                    else:
                        nearest_history.append(None)
                except Exception as e:
                    order_logs.append(f"     → ❌ Échec transfert: {str(e)}")
                    nearest_history.append(None)
            else:
                nearest_history.append(None)

        yield env.timeout(1)

    return satisfied_orders, unsatisfied_orders, total_sales, stock_history, order_logs, nearest_history, mean_demand_daily, std_dev_daily, transfers

def run_simulation_reseau(b):
    with output_reseau:
        clear_output()
        display(dist_dd, brand_dd, fam_dd, reseau_button)

        selected_distributeur = dist_dd.value
        selected_brand = brand_dd.value
        selected_famille = fam_dd.value

        if not selected_brand or not selected_famille:
            print("❌ Veuillez sélectionner une marque et une famille")
            return

        print(f"🔍 Recherche des données pour: {selected_distributeur} - {selected_brand} - {selected_famille}")

        # Filtrer les données
        distributor_data = data[selected_distributeur]
        distributor_data = distributor_data[distributor_data['Brand'] == selected_brand]
        distributor_data = distributor_data[distributor_data['Famille'] == selected_famille]

        if distributor_data.empty:
            print("❌ Aucune donnée disponible pour cette combinaison")
            return

        print(f"✅ Données trouvées pour le distributeur principal")

        # Vérifier que les coordonnées existent
        if selected_distributeur not in data_coords:
            print(f"❌ Coordonnées non trouvées pour {selected_distributeur}")
            print(f"   Distributeurs disponibles: {list(data_coords.keys())}")
            return

        # Lancer la simulation
        env = simpy.Environment()
        process = env.process(supply_chain_reseau(env, distributor_data, data_coords, selected_distributeur, selected_brand, selected_famille))
        env.run()

        satisfied, unsatisfied, total_sales, stock_history, logs, nearest_history, mean_daily, std_daily, transfers = process.value

        # Visualisation
        plt.figure(figsize=(16, 10))

        # Graphique 1: Évolution du stock
        plt.subplot(2, 2, (1, 2))
        plt.plot(stock_history, label=f'Stock - {selected_distributeur}', linewidth=2, color='blue')

        # Lignes de référence
        stock_alert_val = distributor_data['Stock d alerte (TN)'].values[0]
        stock_min_val = distributor_data['Stock min (TN)'].values[0]
        stock_max_val = distributor_data['Stock max (TN)'].values[0]

        plt.axhline(y=stock_alert_val, color='orange', linestyle='--', linewidth=2, label=f'Stock alerte ({stock_alert_val} TN)')
        plt.axhline(y=stock_min_val, color='red', linestyle='--', linewidth=2, label=f'Stock min ({stock_min_val} TN)')
        plt.axhline(y=stock_max_val, color='green', linestyle=':', linewidth=2, label=f'Stock max ({stock_max_val} TN)')

        plt.title(f'📈 Évolution du Stock avec Réseau - {selected_brand} {selected_famille}', fontsize=14)
        plt.xlabel('Jours', fontsize=12)
        plt.ylabel('Stock (TN)', fontsize=12)
        plt.legend(loc='upper right')
        plt.grid(True, alpha=0.3)
        plt.xticks(range(0, 31, 5))

        # Graphique 2: Distributeurs sollicités
        plt.subplot(2, 2, 3)

        # Compter les occurrences
        from collections import Counter
        valid_nearest = [d for d in nearest_history if d is not None]

        if valid_nearest:
            counter = Counter(valid_nearest)
            distributeurs = list(counter.keys())
            counts = list(counter.values())

            colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEEAD', '#D4A5A5']
            bars = plt.bar(distributeurs, counts, color=colors[:len(distributeurs)])
            plt.title('Distributeurs sollicités (nombre de fois)', fontsize=14)
            plt.xlabel('Distributeur', fontsize=12)
            plt.ylabel('Nombre de sollicitations', fontsize=12)
            plt.xticks(rotation=45)

            # Ajouter les valeurs sur les barres
            for bar, count in zip(bars, counts):
                plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                        str(count), ha='center', va='bottom')
        else:
            plt.text(0.5, 0.5, 'Aucun transfert effectué', ha='center', va='center', fontsize=12)
            plt.title('Distributeurs sollicités', fontsize=14)

        plt.grid(True, alpha=0.3)

        # Graphique 3: Carte des distances (simplifiée)
        plt.subplot(2, 2, 4)

        # Créer un graphique en barres des distances
        distances = []
        labels = []

        for dist in data_coords.keys():
            if dist != selected_distributeur:
                try:
                    dist_km = calculate_distance(data_coords[selected_distributeur], data_coords[dist])
                    distances.append(dist_km)
                    labels.append(dist)
                except:
                    pass

        if distances:
            # Trier par distance
            sorted_idx = np.argsort(distances)
            sorted_dist = [distances[i] for i in sorted_idx[:5]]  # Top 5
            sorted_labels = [labels[i] for i in sorted_idx[:5]]

            bars = plt.barh(sorted_labels, sorted_dist, color='lightseagreen')
            plt.title(f'Distances depuis {selected_distributeur} (top 5)', fontsize=14)
            plt.xlabel('Distance (km)', fontsize=12)

            # Ajouter les valeurs
            for bar, dist in zip(bars, sorted_dist):
                plt.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                        f'{dist:.0f} km', va='center')
        else:
            plt.text(0.5, 0.5, 'Données de distance non disponibles', ha='center', va='center')
            plt.title('Distances', fontsize=14)

        plt.tight_layout()
        plt.show()

        # Résultats
        rupture_rate = (unsatisfied / 30) * 100

        # Afficher les résultats dans un cadre stylisé
        from IPython.display import HTML

        # Compter les transferts par distributeur
        transfer_details = ""
        if valid_nearest:
            counter = Counter(valid_nearest)
            transfer_details = "<h4>📊 Détail des transferts:</h4><ul>"
            for dist, count in counter.items():
                transfer_details += f"<li>{dist}: {count} transfert(s)</li>"
            transfer_details += "</ul>"

        result_html = f"""
        <div style="background-color: #f0f8ff; padding: 20px; border-radius: 10px; border: 2px solid #FFA500;">
            <h3 style="color: #333; margin-top: 0;">🌐 RÉSULTATS SIMULATION AVEC RÉSEAU</h3>
            <table style="width: 100%; border-collapse: collapse;">
                <tr style="background-color: #e0e0e0;">
                    <th style="padding: 10px; text-align: left;">Paramètre</th>
                    <th style="padding: 10px; text-align: left;">Valeur</th>
                </tr>
                <tr>
                    <td style="padding: 8px; border-bottom: 1px solid #ddd;"><b>Distributeur principal:</b></td>
                    <td style="padding: 8px; border-bottom: 1px solid #ddd;">{selected_distributeur}</td>
                </tr>
                <tr>
                    <td style="padding: 8px; border-bottom: 1px solid #ddd;"><b>Marque / Famille:</b></td>
                    <td style="padding: 8px; border-bottom: 1px solid #ddd;">{selected_brand} - {selected_famille}</td>
                </tr>
                <tr>
                    <td style="padding: 8px; border-bottom: 1px solid #ddd;"><b>Stock initial:</b></td>
                    <td style="padding: 8px; border-bottom: 1px solid #ddd;">{stock_max_val} TN</td>
                </tr>
                <tr>
                    <td style="padding: 8px; border-bottom: 1px solid #ddd;"><b>Demande moyenne journalière:</b></td>
                    <td style="padding: 8px; border-bottom: 1px solid #ddd;">{mean_daily:.2f} TN</td>
                </tr>
                <tr>
                    <td style="padding: 8px; border-bottom: 1px solid #ddd;"><b>Commandes satisfaites:</b></td>
                    <td style="padding: 8px; border-bottom: 1px solid #ddd;">{satisfied}/30</td>
                </tr>
                <tr>
                    <td style="padding: 8px; border-bottom: 1px solid #ddd;"><b>Commandes non satisfaites:</b></td>
                    <td style="padding: 8px; border-bottom: 1px solid #ddd;">{unsatisfied}/30</td>
                </tr>
                <tr>
                    <td style="padding: 8px; border-bottom: 1px solid #ddd;"><b>Taux de rupture:</b></td>
                    <td style="padding: 8px; border-bottom: 1px solid #ddd;">
                        <span style="background-color: {'#FFB6C1' if rupture_rate > 10 else '#90EE90'}; padding: 3px 8px; border-radius: 5px;">
                            {rupture_rate:.2f}%
                        </span>
                    </td>
                </tr>
                <tr>
                    <td style="padding: 8px; border-bottom: 1px solid #ddd;"><b>Transferts effectués:</b></td>
                    <td style="padding: 8px; border-bottom: 1px solid #ddd;">{transfers}</td>
                </tr>
                <tr>
                    <td style="padding: 8px;"><b>Total ventes:</b></td>
                    <td style="padding: 8px;">{total_sales:.1f} TN</td>
                </tr>
            </table>
            {transfer_details}
        </div>
        """

        display(HTML(result_html))

        # Afficher le journal des événements
        print("\n📋 JOURNAL DES ÉVÉNEMENTS (premiers 15):")
        for log in logs[:15]:
            print(log)

reseau_button.on_click(run_simulation_reseau)
display(reseau_button, output_reseau)

Button(button_style='warning', description='🌐 LANCER LA SIMULATION AVEC RÉSEAU', layout=Layout(height='40px', …

Output()

In [27]:
# Version simplifiée avec diagnostics
predict_simple_btn = widgets.Button(description='📊 PRÉDIRE (version simple)',
                                    button_style='info',
                                    layout=widgets.Layout(width='300px', height='40px'))
output_simple = widgets.Output()

def predict_simple(b):
    with output_simple:
        clear_output()
        display(dist_dd, brand_dd, fam_dd, predict_simple_btn)

        dist = dist_dd.value
        marque = brand_dd.value
        famille = fam_dd.value

        if not all([dist, marque, famille]):
            print("❌ Veuillez faire une sélection complète")
            return

        print(f"\n🔍 PRÉDICTION POUR: {dist} - {marque} - {famille}")
        print("=" * 60)

        # Récupérer les données
        df = data[dist]

        # Liste des mois
        mois = ['Juin', 'Juil', 'Août', 'Sept', 'Oct', 'Nov', 'Déc',
                'Janv', 'Fév', 'Mars', 'Avril', 'Mai']

        # Filtrer
        ligne = df[(df['Brand'] == marque) & (df['Famille'] == famille)]

        if ligne.empty:
            print("❌ Ligne non trouvée")
            return

        # Extraire les ventes
        ventes = []
        for m in mois:
            if m in df.columns:
                ventes.append(ligne[m].values[0])

        if len(ventes) == 0:
            print("❌ Aucune donnée de vente")
            return

        print(f"📊 Données historiques (12 mois):")
        for i, (m, v) in enumerate(zip(mois, ventes)):
            print(f"   {m:5}: {v:6.1f} TN")

        print(f"\n📈 Statistiques descriptives:")
        print(f"   Moyenne: {np.mean(ventes):.1f} TN")
        print(f"   Médiane: {np.median(ventes):.1f} TN")
        print(f"   Écart-type: {np.std(ventes):.1f} TN")
        print(f"   Min: {np.min(ventes):.1f} TN")
        print(f"   Max: {np.max(ventes):.1f} TN")

        # Régression linéaire simple
        X = np.array(range(1, 13)).reshape(-1, 1)
        y = np.array(ventes)

        model = LinearRegression()
        model.fit(X, y)

        # Prédictions
        X_future = np.array(range(13, 25)).reshape(-1, 1)
        y_pred = model.predict(X_future)

        print(f"\n📐 Équation de tendance:")
        print(f"   Ventes = {model.coef_[0]:.2f} × Mois + {model.intercept_:.2f}")
        print(f"   Coefficient de détermination R² = {model.score(X, y):.3f}")

        if model.coef_[0] > 0:
            print(f"   📈 Tendance à la hausse: +{model.coef_[0]:.2f} TN par mois")
        else:
            print(f"   📉 Tendance à la baisse: {model.coef_[0]:.2f} TN par mois")

        # Graphique
        plt.figure(figsize=(12, 6))

        plt.plot(range(1, 13), ventes, 'o-', label='Historique', linewidth=2, markersize=8)
        plt.plot(range(13, 25), y_pred, 's--', label='Prédiction', linewidth=2, markersize=8)

        plt.axhline(y=np.mean(ventes), color='gray', linestyle=':', label=f'Moyenne: {np.mean(ventes):.1f}')

        plt.title(f'Prédiction des ventes - {marque} {famille}', fontsize=14)
        plt.xlabel('Mois')
        plt.ylabel('Ventes (TN)')
        plt.xticks(range(1, 25), mois*2, rotation=45)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # Résumé
        print(f"\n📋 RÉSUMÉ DE LA PRÉDICTION:")
        print(f"   Total année 1 (historique): {sum(ventes):.1f} TN")
        print(f"   Total année 2 (prédite): {sum(y_pred):.1f} TN")
        print(f"   Évolution: {((sum(y_pred) - sum(ventes)) / sum(ventes) * 100):+.1f}%")

predict_simple_btn.on_click(predict_simple)
display(predict_simple_btn, output_simple)

Button(button_style='info', description='📊 PRÉDIRE (version simple)', layout=Layout(height='40px', width='300p…

Output()

In [30]:
# ============================================
# ÉTAPE 1 : S'ASSURER QUE TOUS LES BOUTONS SONT DÉFINIS
# ============================================

# Bouton pour l'analyse statistique
analyse_button = widgets.Button(
    description='📊 ANALYSER LA DISTRIBUTION',
    button_style='info',
    layout=widgets.Layout(width='300px', height='40px')
)
output_analyse = widgets.Output()

# Bouton pour la simulation simple
simulation_button = widgets.Button(
    description='📈 SIMULATION SIMPLE',
    button_style='primary',
    layout=widgets.Layout(width='300px', height='40px')
)
output_simulation = widgets.Output()

# Bouton pour la simulation réseau
reseau_button = widgets.Button(
    description='🌐 SIMULATION RÉSEAU',
    button_style='warning',
    layout=widgets.Layout(width='300px', height='40px')
)
output_reseau = widgets.Output()

# Bouton pour la prédiction
predict_button = widgets.Button(
    description='📉 PRÉDIRE LES VENTES',
    button_style='success',
    layout=widgets.Layout(width='300px', height='40px')
)
output_predict = widgets.Output()

print("✅ Tous les boutons ont été créés avec succès!")
print(f"   - analyse_button: {type(analyse_button)}")
print(f"   - simulation_button: {type(simulation_button)}")
print(f"   - reseau_button: {type(reseau_button)}")
print(f"   - predict_button: {type(predict_button)}")

✅ Tous les boutons ont été créés avec succès!
   - analyse_button: <class 'ipywidgets.widgets.widget_button.Button'>
   - simulation_button: <class 'ipywidgets.widgets.widget_button.Button'>
   - reseau_button: <class 'ipywidgets.widgets.widget_button.Button'>
   - predict_button: <class 'ipywidgets.widgets.widget_button.Button'>


In [31]:
# ============================================
# FONCTION POUR L'ANALYSE STATISTIQUE
# ============================================

def on_analyse_click(b):
    with output_analyse:
        clear_output()
        display(dist_dd, brand_dd, fam_dd, analyse_button)

        selected_distributeur = dist_dd.value
        selected_brand = brand_dd.value
        selected_famille = fam_dd.value

        if not selected_brand or not selected_famille:
            print("❌ Veuillez sélectionner une marque et une famille")
            return

        print(f"📊 Analyse pour: {selected_distributeur} - {selected_brand} - {selected_famille}")

        # Récupérer les données
        df = data[selected_distributeur]

        # Mois
        mois = ['Juin', 'Juil', 'Août', 'Sept', 'Oct', 'Nov', 'Déc',
                'Janv', 'Fév', 'Mars', 'Avril', 'Mai']

        # Filtrer les colonnes qui existent
        mois_existants = [m for m in mois if m in df.columns]

        if not mois_existants:
            print("❌ Colonnes de ventes non trouvées")
            return

        # Filtrer les données
        mask = (df['Brand'] == selected_brand) & (df['Famille'] == selected_famille)
        ventes = df.loc[mask, mois_existants]

        if ventes.empty:
            print("❌ Aucune donnée trouvée")
            return

        # Aplatir les données
        valeurs = ventes.values.flatten()

        # Créer les graphiques
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))

        # Histogramme
        axes[0].hist(valeurs, bins=8, color='skyblue', edgecolor='black', alpha=0.7)
        axes[0].axvline(valeurs.mean(), color='red', linewidth=2, label=f'Moyenne: {valeurs.mean():.1f}')
        axes[0].set_title(f'Distribution des ventes', fontsize=12)
        axes[0].set_xlabel('Ventes (TN)')
        axes[0].set_ylabel('Fréquence')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Q-Q Plot
        stats.probplot(valeurs, dist="norm", plot=axes[1])
        axes[1].set_title('Q-Q Plot (Test de normalité)', fontsize=12)
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        # Test de Shapiro-Wilk
        if 3 <= len(valeurs) <= 5000:
            stat, p = stats.shapiro(valeurs)
            print(f"\n🔬 TEST DE SHAPIRO-WILK:")
            print(f"   • Statistique W: {stat:.6f}")
            print(f"   • p-value: {p:.6f}")
            if p > 0.05:
                print("   ✅ Conclusion: Distribution normale (p > 0.05)")
            else:
                print("   ❌ Conclusion: Distribution non normale (p < 0.05)")

analyse_button.on_click(on_analyse_click)

# ============================================
# FONCTION POUR LA SIMULATION SIMPLE
# ============================================

def simulate_order(mean_demand, std_dev):
    quantity = abs(np.random.normal(mean_demand, std_dev))
    return max(1, int(np.round(quantity)))

def on_simulation_click(b):
    with output_simulation:
        clear_output()
        display(dist_dd, brand_dd, fam_dd, simulation_button)

        selected_distributeur = dist_dd.value
        selected_brand = brand_dd.value
        selected_famille = fam_dd.value

        if not selected_brand or not selected_famille:
            print("❌ Veuillez sélectionner une marque et une famille")
            return

        # Filtrer les données
        df = data[selected_distributeur]
        data_row = df[(df['Brand'] == selected_brand) & (df['Famille'] == selected_famille)]

        if data_row.empty:
            print("❌ Aucune donnée disponible")
            return

        # Paramètres
        stock_max = data_row['Stock max (TN)'].values[0]
        stock_alerte = data_row['Stock d alerte (TN)'].values[0]
        stock_min = data_row['Stock min (TN)'].values[0]
        demande_moy = data_row['demande moyenne'].values[0] / 30
        ecart_type = data_row['Ecart-type'].values[0] / np.sqrt(30)

        print(f"📊 Simulation démarrée pour {selected_brand} - {selected_famille}")

        # Simulation
        stock = stock_max
        historique = []
        satisfaites = 0
        non_satisfaites = 0
        ventes_total = 0

        for jour in range(30):
            historique.append(stock)

            # Réapprovisionnement
            if jour > 0 and jour % 7 == 0:
                stock = stock_max

            # Commande
            cmd = simulate_order(demande_moy, ecart_type)

            if cmd <= stock:
                satisfaites += 1
                ventes_total += cmd
                stock -= cmd
            else:
                non_satisfaites += 1

        # Graphique
        plt.figure(figsize=(12, 6))
        plt.plot(historique, linewidth=2, color='blue')
        plt.axhline(y=stock_alerte, color='orange', linestyle='--', label=f'Alerte ({stock_alerte})')
        plt.axhline(y=stock_min, color='red', linestyle='--', label=f'Min ({stock_min})')
        plt.title(f'Simulation - {selected_brand} {selected_famille}')
        plt.xlabel('Jours')
        plt.ylabel('Stock')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

        # Résultats
        print(f"\n✅ Commandes satisfaites: {satisfaites}/30")
        print(f"❌ Commandes non satisfaites: {non_satisfaites}/30")
        print(f"📊 Taux de rupture: {(non_satisfaites/30)*100:.1f}%")
        print(f"💰 Total ventes: {ventes_total:.1f} TN")

simulation_button.on_click(on_simulation_click)

# ============================================
# FONCTION POUR LA SIMULATION RÉSEAU
# ============================================

def on_reseau_click(b):
    with output_reseau:
        clear_output()
        display(dist_dd, brand_dd, fam_dd, reseau_button)

        selected_distributeur = dist_dd.value
        selected_brand = brand_dd.value
        selected_famille = fam_dd.value

        if not selected_brand or not selected_famille:
            print("❌ Veuillez sélectionner une marque et une famille")
            return

        print(f"🌐 Simulation réseau pour {selected_distributeur}")
        print("⚠️ Version simplifiée - À compléter selon vos besoins")

        # Placeholder pour la simulation réseau
        plt.figure(figsize=(10, 6))
        plt.text(0.5, 0.5, 'Simulation réseau en cours de développement',
                ha='center', va='center', fontsize=14)
        plt.show()

reseau_button.on_click(on_reseau_click)

# ============================================
# FONCTION POUR LA PRÉDICTION
# ============================================

def on_predict_click(b):
    with output_predict:
        clear_output()
        display(dist_dd, brand_dd, fam_dd, predict_button)

        selected_distributeur = dist_dd.value
        selected_brand = brand_dd.value
        selected_famille = fam_dd.value

        if not selected_brand or not selected_famille:
            print("❌ Veuillez sélectionner une marque et une famille")
            return

        print(f"📈 Prédiction pour {selected_brand} - {selected_famille}")

        # Récupérer les données
        df = data[selected_distributeur]
        mois = ['Juin', 'Juil', 'Août', 'Sept', 'Oct', 'Nov', 'Déc',
                'Janv', 'Fév', 'Mars', 'Avril', 'Mai']

        mois_existants = [m for m in mois if m in df.columns]

        if not mois_existants:
            print("❌ Colonnes de ventes non trouvées")
            return

        # Filtrer
        mask = (df['Brand'] == selected_brand) & (df['Famille'] == selected_famille)
        ventes = df.loc[mask, mois_existants].values.flatten()

        if len(ventes) == 0:
            print("❌ Aucune donnée")
            return

        # Régression
        X = np.array(range(1, 13)).reshape(-1, 1)
        y = ventes

        model = LinearRegression()
        model.fit(X, y)

        # Prédiction
        X_futur = np.array(range(13, 25)).reshape(-1, 1)
        y_pred = model.predict(X_futur)

        # Graphique
        plt.figure(figsize=(14, 6))
        plt.plot(range(1, 13), ventes, 'o-', label='Historique', linewidth=2, markersize=8)
        plt.plot(range(13, 25), y_pred, 's--', label='Prédiction', linewidth=2, markersize=8, color='red')
        plt.title(f'Prédiction - {selected_brand} {selected_famille}')
        plt.xlabel('Mois')
        plt.ylabel('Ventes')
        plt.xticks(range(1, 25), mois*2, rotation=45)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        print(f"\n📊 Équation: Ventes = {model.coef_[0]:.2f} × Mois + {model.intercept_:.2f}")
        print(f"📈 Total prédit année 2: {sum(y_pred):.1f} TN")

predict_button.on_click(on_predict_click)

print("✅ Toutes les fonctions ont été attachées aux boutons!")

✅ Toutes les fonctions ont été attachées aux boutons!


In [32]:
# ============================================
# TABLEAU DE BORD
# ============================================

# Créer le tableau de bord
dashboard = widgets.Tab(layout=widgets.Layout(width='100%', min_height='600px'))

# Contenu des onglets
contenu = [
    # Onglet Sélection
    widgets.VBox([
        widgets.HTML("<h3 style='color: #4CAF50;'>🔍 SÉLECTION DES PARAMÈTRES</h3>"),
        widgets.HTML("<p><i>Choisissez les éléments à analyser :</i></p>"),
        widgets.VBox([dist_dd, brand_dd, fam_dd], layout=widgets.Layout(padding='20px', background_color='#f5f5f5'))
    ]),

    # Onglet Analyse
    widgets.VBox([
        widgets.HTML("<h3 style='color: #2196F3;'>📊 ANALYSE STATISTIQUE</h3>"),
        widgets.HTML("<p><i>Test de normalité et distribution</i></p>"),
        analyse_button,
        output_analyse
    ]),

    # Onglet Simulation simple
    widgets.VBox([
        widgets.HTML("<h3 style='color: #FF9800;'>📈 SIMULATION SIMPLE</h3>"),
        widgets.HTML("<p><i>Gestion de stock basique</i></p>"),
        simulation_button,
        output_simulation
    ]),

    # Onglet Simulation réseau
    widgets.VBox([
        widgets.HTML("<h3 style='color: #9C27B0;'>🌐 SIMULATION RÉSEAU</h3>"),
        widgets.HTML("<p><i>Avec transferts entre distributeurs</i></p>"),
        reseau_button,
        output_reseau
    ]),

    # Onglet Prédiction
    widgets.VBox([
        widgets.HTML("<h3 style='color: #E91E63;'>📉 PRÉDICTION</h3>"),
        widgets.HTML("<p><i>Prévision des ventes sur 12 mois</i></p>"),
        predict_button,
        output_predict
    ])
]

# Assigner les onglets
dashboard.children = contenu
dashboard.set_title(0, '🔍 Sélection')
dashboard.set_title(1, '📊 Analyse')
dashboard.set_title(2, '📈 Simple')
dashboard.set_title(3, '🌐 Réseau')
dashboard.set_title(4, '📉 Prédiction')

# Afficher
display(dashboard)

# Barre d'état
from IPython.display import HTML

status_html = f"""
<div style='background-color: #e3f2fd; padding: 15px; border-radius: 5px; margin-top: 20px;'>
    <b>✅ TABLEAU DE BORD PRÊT</b><br>
    <span>Distributeur: <b style='color: #4CAF50;'>{dist_dd.value if dist_dd.value else 'Non sélectionné'}</b></span><br>
    <span>Marque: <b style='color: #2196F3;'>{brand_dd.value if brand_dd.value else 'Non sélectionnée'}</b></span><br>
    <span>Famille: <b style='color: #FF9800;'>{fam_dd.value if fam_dd.value else 'Non sélectionnée'}</b></span>
</div>
"""

display(HTML(status_html))
print("\n👆 Sélectionnez d'abord un distributeur, une marque et une famille dans le premier onglet")


👆 Sélectionnez d'abord un distributeur, une marque et une famille dans le premier onglet
